In [ ]:
# ==========================================
# V.4 TIEN XU LY & FEATURE ENGINEERING: TF-IDF tren review_comment_message
# ==========================================

import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import joblib

# tai stopwords tieng Bo Dao Nha
nltk.download('stopwords', quiet=True)
stop_words_pt = stopwords.words('portuguese')

# doc du lieu
df = pd.read_parquet('master_dataset.parquet')
print('Shape du lieu goc:', df.shape)
print('So cot:', len(df.columns))

# kiem tra cot dau vao bat buoc
required_col = 'review_comment_message'
if required_col not in df.columns:
    raise ValueError("Khong tim thay cot 'review_comment_message' trong dataset")

# kiem thu du lieu truoc khi dua vao TF-IDF
missing_cnt = int(df[required_col].isna().sum())
empty_cnt = int((df[required_col].fillna('').astype(str).str.strip() == '').sum())
print(f"So dong null o {required_col}: {missing_cnt}")
print(f"So dong rong o {required_col}: {empty_cnt}")
print(f"Ty le rong/null: {(empty_cnt / len(df)):.2%}")

# xu ly null (giu lai toan bo dong)
df_reviews = df.copy()
df_reviews[required_col] = df_reviews[required_col].fillna('').astype(str)

# EDA nhanh cho text: phan phoi do dai review
df_reviews['review_len'] = df_reviews[required_col].str.len()
print('\nThong ke do dai review (range/phan phoi):')
print(df_reviews['review_len'].describe().to_string())

# xem mau du lieu truoc khi vectorize
print('\n5 mau review truoc xu ly TF-IDF:')
print(df_reviews[required_col].head(5).to_string(index=False))

# chia train/test 80/20 voi seed co dinh de tai lap ket qua
train_df, test_df = train_test_split(
    df_reviews,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# Pipeline TF-IDF (fit/transform + save/load)
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stop_words_pt,
        max_features=500,
        lowercase=True,
        min_df=5,
        token_pattern=r'(?u)\b[a-zA-ZÀ-ÿ]{2,}\b'
    ))
])

# fit tren train, transform ca train/test
X_train_tfidf = tfidf_pipeline.fit_transform(train_df[required_col])
X_test_tfidf = tfidf_pipeline.transform(test_df[required_col])

# thong tin kiem tra ti le 80/20 + kich thuoc ma tran
n_total = len(df_reviews)
n_train = len(train_df)
n_test = len(test_df)
print(f"\nTotal: {n_total} | Train: {n_train} ({n_train / n_total:.2%}) | Test: {n_test} ({n_test / n_total:.2%})")
print('Kich thuoc ma tran TF-IDF train:', X_train_tfidf.shape)
print('Kich thuoc ma tran TF-IDF test:', X_test_tfidf.shape)
print('Top 20 tu khoa:', tfidf_pipeline.named_steps['tfidf'].get_feature_names_out()[:20])

# luu va load lai pipeline TF-IDF
joblib.dump(tfidf_pipeline, 'tfidf_pipeline.joblib')
loaded_pipeline = joblib.load('tfidf_pipeline.joblib')
_ = loaded_pipeline.transform(test_df[required_col].head(3))
print('Da luu & load lai tfidf_pipeline.joblib thanh cong')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Kích thước ma trận TF-IDF: (113425, 500)
Top 20 từ khóa: ['abri' 'abrir' 'absurdo' 'acabamento' 'achei' 'acho' 'aconteceu' 'acordo'
 'acredito' 'adorei' 'agilidade' 'agora' 'agradeço' 'aguardando' 'aguardo'
 'ainda' 'algum' 'alguns' 'além' 'amei']
✅ Đã lưu tfidf_vectorizer.joblib


In [ ]:
# ==========================================
# VI.5 TRIỂN KHAI MÔ HÌNH: FP-GROWTH CHO GIỎ HÀNG
# ==========================================

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# đọc dữ liệu
df = pd.read_parquet("master_dataset.parquet")
print("Shape dữ liệu gốc:", df.shape)

# kiểm tra cột đầu vào bắt buộc
cat_candidates = ["product_category_name_english", "product_category_name"]
cat_col = next((c for c in cat_candidates if c in df.columns), None)

if "order_id" not in df.columns:
    raise ValueError("Không tìm thấy cột 'order_id' trong dataset")
if cat_col is None:
    raise ValueError("Không tìm thấy cột category (product_category_name_english hoặc product_category_name)")

print("Cột category dùng cho FP-Growth:", cat_col)

# kiểm thử dữ liệu trước khi đưa vào FP-Growth
missing_order = int(df["order_id"].isna().sum())
missing_cat = int(df[cat_col].isna().sum())
print(f"Số dòng thiếu order_id: {missing_order}")
print(f"Số dòng thiếu {cat_col}: {missing_cat}")
print(f"Tỷ lệ thiếu {cat_col}: {(missing_cat / len(df)):.2%}")

dup_pairs = int(df[["order_id", cat_col]].dropna().duplicated().sum())
print("Số bản ghi trùng (order_id, category):", dup_pairs)

print("\n5 dòng dữ liệu gốc cho FP-Growth:")
print(df[["order_id", cat_col]].head(5).to_string(index=False))

# tiền xử lý: drop null và chuẩn hóa text category
df_items = df.dropna(subset=["order_id", cat_col]).copy()
df_items[cat_col] = df_items[cat_col].astype(str).str.strip()
df_items = df_items[df_items[cat_col] != ""]

print("\nSố dòng sau làm sạch:", len(df_items))
print("Số order duy nhất:", df_items["order_id"].nunique())
print("Số category duy nhất:", df_items[cat_col].nunique())

# tạo transactions theo order
transactions = (
    df_items.groupby("order_id")[cat_col]
    .apply(lambda x: sorted(set(x)))
    .tolist()
)

# EDA nhanh: phân phối số item mỗi giao dịch
txn_len = pd.Series([len(t) for t in transactions], name="items_per_order")
print("\nThống kê số item mỗi order (trước lọc):")
print(txn_len.describe().to_string())

# lọc đơn có >=2 sản phẩm
transactions = [t for t in transactions if len(t) >= 2]
print("Đơn hàng có >=2 sản phẩm:", len(transactions))

if len(transactions) == 0:
    raise ValueError("Không có transaction nào đủ điều kiện (>=2 sản phẩm) để chạy FP-Growth")

# one-hot transaction matrix
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)
print("Kích thước ma trận transaction one-hot:", df_trans.shape)

# FP-Growth
frequent_itemsets = fpgrowth(df_trans, min_support=0.0001, use_colnames=True)
print("Frequent itemsets:", frequent_itemsets.shape)

if frequent_itemsets.empty:
    raise ValueError("Không tìm được frequent itemsets. Hãy tăng dữ liệu hoặc giảm min_support.")

# lưu frequent itemsets để làm minh chứng trong báo cáo
fi_export = frequent_itemsets.copy()
fi_export["itemsets"] = fi_export["itemsets"].apply(lambda x: ", ".join(sorted(list(x))))
fi_export.to_csv("frequent_itemsets.csv", index=False)
print("Đã lưu frequent_itemsets.csv")

# sinh luật kết hợp
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.01)
print("Rules:", rules.shape)

if rules.empty:
    print("Không có luật nào thỏa confidence hiện tại. Hãy giảm min_threshold.")
    top_rules = pd.DataFrame(columns=["antecedents", "consequents", "support", "confidence", "lift"])
else:
    # lưu toàn bộ rules
    rules_export = rules.copy()
    rules_export["antecedents"] = rules_export["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
    rules_export["consequents"] = rules_export["consequents"].apply(lambda x: ", ".join(sorted(list(x))))
    rules_export.to_csv("association_rules_full.csv", index=False)
    print("Đã lưu association_rules_full.csv")

    # preview top 10 theo lift và confidence
    top_rules = rules.sort_values(by=["lift", "confidence"], ascending=False).head(10).copy()
    top_rules["antecedents"] = top_rules["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
    top_rules["consequents"] = top_rules["consequents"].apply(lambda x: ", ".join(sorted(list(x))))

print("\nTop 10 rules:")
print(top_rules[["antecedents", "consequents", "support", "confidence", "lift"]].to_string(index=False))

top_rules.to_csv("fp_growth_rules.csv", index=False)
print("Đã lưu fp_growth_rules.csv")
# Sắp xếp luật theo lift rồi confidence, lấy Top 5
top5_rules = (
    rules.sort_values(by=['lift', 'confidence'], ascending=False)
        .head(5)
        .copy()
)

# Chuyển frozenset sang chuỗi cho dễ đọc trong báo cáo
top5_rules['antecedents'] = top5_rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
top5_rules['consequents'] = top5_rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

# Chỉ giữ đúng các cột cần nộp
top5_rules = top5_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

# Làm đẹp số liệu
top5_rules[['support', 'confidence', 'lift']] = top5_rules[['support', 'confidence', 'lift']].round(4)

print('Top 5 luật kết hợp:')
print(top5_rules.to_string(index=False))

# Nếu cần đưa vào Word/Excel
top5_rules.to_csv('top5_association_rules.csv', index=False)